<h1 align="center"> Read and Analyze Submissions Data Set</h1>

### Import Code Libraries

In [ ]:
import polars as pl
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import display, HTML

%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))

### Read 2022 Q1 Submission Data Set from [Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets) into a Polars DataFrame named filingsQ1

In [ ]:
# scan_csv is lazy (nothing loads until .collect() is called)
filingsQ1 = (
    pl.scan_csv('data/2022q1/sub.txt', separator='\t', infer_schema_length=10000)
    .collect()
)

### Look at the contents of filingsQ1 DataFrame. We have 7,237 submissions with 36 columns.

In [ ]:
filingsQ1

### View the first two rows of filingsQ1

In [ ]:
filingsQ1.head(2)

### View the last three rows of filingsQ1

In [ ]:
filingsQ1.tail(3)

### View rows 100–101 of filingsQ1

In [ ]:
filingsQ1[100:102]

### List All Column Names of the DataFrame

In [ ]:
filingsQ1.columns

### Create a New DataFrame with a Subset of Columns

In [ ]:
subsetDF = filingsQ1.select(['name', 'countryinc', 'afs', 'form', 'period', 'fy', 'fp', 'filed'])
subsetDF

### Print DataFrame Schema (Data Types) and Shape

In [ ]:
print(subsetDF.schema)
print('Shape:', subsetDF.shape)

### Generate Descriptive Statistics

In [ ]:
subsetDF.describe()

### Find Unique Values of countryinc Column

In [ ]:
subsetDF['countryinc'].unique()

### Filter for Canadian Filers

In [ ]:
subsetDF.filter(pl.col('countryinc') == 'CA')

### Group Submissions By Form Column and Compute Group Sizes

In [ ]:
formGroups = (
    subsetDF
    .group_by('form')
    .agg(pl.len().alias('count'))
    .sort('count')
)
formGroups

### Filter Submissions for Important Forms (10-K, 10-Q, 20-F, 40-F)

In [ ]:
importantForms = ['10-K', '10-K/A', '10-Q', '10-Q/A', '20-F', '20-F/A', '40-F', '40-F/A']
importantFormGroups = formGroups.filter(pl.col('form').is_in(importantForms))
importantFormGroups

### Visualize Statistics for Important Forms using Pyplot Library

In [ ]:
importantFormGroups.to_pandas().plot(x='form', y='count', kind='bar', figsize=(8,6))
plt.title('Structured Data Form Statistics for 2022 Q1', fontsize=15)
plt.xlabel('Form Type', fontsize=15)
plt.ylabel('Number of Filings', fontsize=15)
plt.yticks(np.arange(0, 5001, step=500))
plt.xticks(rotation=0)
plt.savefig('images/FormStats2022Q1.jpg')
plt.show()